In [ ]:
!export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/home/dongyoonhwang/.mujoco/mujoco210/bin:/usr/lib/nvidia
!export MUJOCO_PY_MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJLIB_PATH=/home/nas2_userI/byungkunlee/.mujoco/mujoco210/bin/libmujoco210.so
!export MUJOCO_PY_MUJOCO_PATH=/home/dongyoonhwang/.mujoco/mujoco210

# >>> MuJoCo visualization >>>
!export XLA_PYTHON_CLIENT_PREALLOCATE=false
!export MUJOCO_GL='egl'
!export MUJOCO_EGL_DEVICE_ID='0'
!export MKL_SERVICE_FORCE_INTEL='0'

In [1]:
import sys
import os
import math
import pandas as pd
sys.path.append("../")
sys.path.append("../..")

In [2]:
import numpy as np
from rliable import library as rly
from rliable import metrics as rly_metrics
from rliable import plot_utils as rly_plot_utils

aggregate_func = lambda x: np.array([
  rly_metrics.aggregate_iqm(x),
  rly_metrics.aggregate_median(x),
  rly_metrics.aggregate_mean(x)])

In [3]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE
from scale_rl.envs.dmc import DMC_EASY_MEDIUM, DMC_HARD
from scale_rl.envs.humanoid_bench import HB_LOCOMOTION_NOHAND, HB_RANDOM_SCORE, HB_SUCCESS_SCORE
from scale_rl.envs.myosuite import MYOSUITE_TASKS

def replace_hypen_to_underbar(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
def replace_hypen_to_underbar_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

MUJOCO_ALL = replace_hypen_to_underbar(MUJOCO_ALL)
DMC_EASY_MEDIUM = replace_hypen_to_underbar(DMC_EASY_MEDIUM)
DMC_HARD = replace_hypen_to_underbar(DMC_HARD)
HB_LOCOMOTION_NOHAND = replace_hypen_to_underbar(HB_LOCOMOTION_NOHAND)
MYOSUITE_TASKS = replace_hypen_to_underbar(MYOSUITE_TASKS)

HB_SUCCESS_SCORE = replace_hypen_to_underbar_dict(HB_SUCCESS_SCORE)
HB_RANDOM_SCORE = replace_hypen_to_underbar_dict(HB_RANDOM_SCORE)

DMC_ALL = [*DMC_EASY_MEDIUM, *DMC_HARD]

/home/nas4_user/youngdolee/anaconda3/envs/scale_rl/lib/python3.9/site-packages/glfw/__init__.py:917: GLFWError: (65550) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


In [4]:
from scale_rl.common.wandb_utils import *

/home/nas4_user/youngdolee/anaconda3/envs/scale_rl/lib/python3.9/site-packages/wandb/analytics/sentry.py:90: SentryHubDeprecationWarning: `sentry_sdk.Hub` is deprecated and will be removed in a future major release. Please consult our 1.x to 2.x migration guide for details on how to migrate `Hub` usage to the new API: https://docs.sentry.io/platforms/python/migration/1.x-to-2.x
  self.hub = sentry_sdk.Hub(client)
/home/nas4_user/youngdolee/anaconda3/envs/scale_rl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
eval_df = pd.read_csv('/home/nas4_user/youngdolee/SimbaV2/results/baseline/sac.csv', index_col=0)
eval_df['exp_name'] = 'SAC'
eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')
eval_df = eval_df[eval_df['env_step'] <= 1e6]
mujoco_eval_df = eval_df[eval_df['env_name'].isin(MUJOCO_ALL)].sort_values("env_name")
total_env_steps = min(max(mujoco_eval_df['env_step']), 1e6)
print(max(mujoco_eval_df['env_step']))
metric = "avg_return"

999960.0


In [7]:
eval_df = mujoco_eval_df

In [8]:
experiments = eval_df["exp_name"].unique()
environments = eval_df["env_name"].unique()

metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=eval_df, 
        random_score_dict=MUJOCO_RANDOM_SCORE,
        base_score_dict=MUJOCO_TD3_SCORE,
    ),
    env_step=total_env_steps,
    metric_type=metric,
)

aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)
print(aggregate_scores)

eval_df = eval_df[eval_df["metric"] == metric]
eval_df = eval_df[eval_df["env_step"] <= total_env_steps]

mujoco_full_results_df = pd.DataFrame(columns=["Task", "SAC"])

for _exp_name in experiments:
    _exp_data = eval_df[eval_df["exp_name"] == _exp_name]
    _num_seeds = _exp_data["seed"].nunique()
    print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
    
for i, env in enumerate(environments):
    mean_ci = []
    env_data = eval_df[eval_df["env_name"] == env]
    print(env, env_data["seed"].nunique())
    for j, exp in enumerate(experiments):
        exp_data = env_data[env_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        
        assert max(exp_data["env_step"]) == total_env_steps
        exp_data = exp_data[exp_data["env_step"] == total_env_steps]
        num_seeds = exp_data["seed"].nunique()
        
        grouped_data = exp_data.groupby("env_step")["value"]

        env_steps = grouped_data.mean().index.values
        mean = float(grouped_data.mean().values)
        std_err = float(grouped_data.sem().values)

        # Plot mean history
        low_CI = mean - 1.960 * std_err
        high_CI = mean + 1.960 * std_err
        
        if metric == "avg_success":
            mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
        else:    
            mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    mujoco_full_results_df = pd.concat([mujoco_full_results_df, pd.DataFrame.from_dict(
                            data=[{"Task": "\\texttt{" + str(env) + "}",
                            "SAC": mean_ci[0],
                            }], orient='columns')], ignore_index=True)


for i, agg in enumerate(["IQM", "Median", "Mean"]):
    mean_ci = []
    for j, exp in enumerate(experiments):
        mean = aggregate_scores[exp][i]
        low_CI = aggregate_score_cis[exp][0][i]
        high_CI = aggregate_score_cis[exp][1][i]
        
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    mujoco_full_results_df = pd.concat([mujoco_full_results_df, pd.DataFrame.from_dict(
                        data=[{"Task": agg,
                            "SAC": mean_ci[0],
                        }], orient='columns')], ignore_index=True)

mujoco_full_results_df

{'SAC': array([1.0971628 , 1.09257708, 1.09218717])}
exp_name: SAC - num_seeds: 10
Ant-v4 10
HalfCheetah-v4 10
Hopper-v4 10
Humanoid-v4 10
Walker2d-v4 10


,Task,SAC
0,\texttt{Ant-v4},"5733 \textcolor{gray}{[5316, 6151]}"
1,\texttt{HalfCheetah-v4},"11320 \textcolor{gray}{[10634, 12007]}"
2,\texttt{Hopper-v4},"2787 \textcolor{gray}{[2249, 3325]}"
3,\texttt{Humanoid-v4},"4825 \textcolor{gray}{[3784, 5866]}"
4,\texttt{Walker2d-v4},"4536 \textcolor{gray}{[4229, 4843]}"
5,IQM,"1.097 \textcolor{gray}{[1.05, 1.155]}"
6,Median,"1.093 \textcolor{gray}{[1.028, 1.18]}"
7,Mean,"1.092 \textcolor{gray}{[1.013, 1.166]}"
